# Assignment 2: Portfolio Optimization using SCS Solver

#### Objective: Construct an optimal portfolio from Nifty 50 assets using the SCS (Splitting Conic Solver)

### Imports

In [1]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import shutil
import os
import cvxpy as cp
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

### Download Data

In [10]:
# 1. Download Data via KaggleHub
print("Downloading Nifty 50 data...")
cache_path = kagglehub.dataset_download("rohanrao/nifty50-stock-market-data")

# 2. Move to local folder (flattening structure)
source_path = Path(cache_path)
if source_path.is_file():
    shutil.copy2(source_path, DATA_DIR / source_path.name)
elif source_path.is_dir():
    for f in source_path.glob("**/*.csv"): # Recursive search
        shutil.copy2(f, DATA_DIR / f.name)


print("Reading CSV files...")
files = list(DATA_DIR.glob("*.csv"))
print(f"Found {len(files)} CSV files.")
raw_df = pd.read_csv(Path.joinpath(DATA_DIR,"NIFTY50_all.csv"))

Reading CSV files...
Found 52 CSV files.


In [11]:
raw_df.describe(include='all')

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
count,235192,235192,235192,235192.000000,235192.000000,235192.000000,235192.000000,235192.000000,235192.000000,235192.00000,2.351920e+05,2.351920e+05,1.203440e+05,2.191150e+05,219115.000000
unique,5306,65,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2021-04-30,ASIANPAINT,EQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,49,5306,235192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,1266.196349,1267.759708,1286.581440,1247.488465,1266.388302,1266.554351,1267.13230,3.045903e+06,1.610138e+14,6.196427e+04,1.315098e+06,0.502997
std,NaN,NaN,NaN,2581.370320,2585.259609,2619.649216,2546.621396,2581.392543,2582.140942,2582.69998,7.333981e+06,3.298085e+14,6.866457e+04,2.831670e+06,0.190019
min,NaN,NaN,NaN,0.000000,8.500000,9.750000,8.500000,9.100000,9.150000,9.21000,3.000000e+00,1.047000e+07,1.100000e+01,5.000000e+00,0.023600
25%,NaN,NaN,NaN,274.300000,275.000000,279.500000,269.600000,274.400000,274.350000,274.69750,2.190095e+05,1.612816e+13,2.183400e+04,1.253830e+05,0.364700
50%,NaN,NaN,NaN,566.500000,567.025000,576.900000,556.500000,567.000000,566.700000,566.94000,1.010938e+06,6.832603e+13,4.406800e+04,5.017560e+05,0.511000
75%,NaN,NaN,NaN,1242.200000,1243.312500,1263.000000,1221.650000,1242.900000,1242.400000,1242.66250,3.019851e+06,1.863835e+14,7.893550e+04,1.452233e+06,0.638400


In [13]:
unique_symbols = raw_df['Symbol'].unique()
unique_symbols.sort()

print(f"Total Unique Assets: {len(unique_symbols)}")

for ticker in unique_symbols:
    print(ticker)

Total Unique Assets: 65
ADANIPORTS
ASIANPAINT
AXISBANK
BAJAJ-AUTO
BAJAJFINSV
BAJAUTOFIN
BAJFINANCE
BHARTI
BHARTIARTL
BPCL
BRITANNIA
CIPLA
COALINDIA
DRREDDY
EICHERMOT
GAIL
GRASIM
HCLTECH
HDFC
HDFCBANK
HEROHONDA
HEROMOTOCO
HINDALC0
HINDALCO
HINDLEVER
HINDUNILVR
ICICIBANK
INDUSINDBK
INFOSYSTCH
INFY
IOC
ITC
JSWSTEEL
JSWSTL
KOTAKBANK
KOTAKMAH
LT
M&M
MARUTI
MUNDRAPORT
NESTLEIND
NTPC
ONGC
POWERGRID
RELIANCE
SBIN
SESAGOA
SHREECEM
SSLT
SUNPHARMA
TATAMOTORS
TATASTEEL
TCS
TECHM
TELCO
TISCO
TITAN
ULTRACEMCO
UNIPHOS
UPL
UTIBANK
VEDL
WIPRO
ZEEL
ZEETELE


In [14]:
raw_df.isna().sum()

Date                       0
Symbol                     0
Series                     0
Prev Close                 0
Open                       0
High                       0
Low                        0
Last                       0
Close                      0
VWAP                       0
Volume                     0
Turnover                   0
Trades                114848
Deliverable Volume     16077
%Deliverble            16077
dtype: int64